# Baseline zero-shot Qwen3-VL 2B su IU X-Ray italiano

Questo notebook calcola l'inferenza zero-shot sullo stesso test set usato per confrontare LoRA e QLoRA. Non esegue fine-tuning.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Esegui il notebook dal repository SHIELD o da una sua sottocartella.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIG_PATH = "configs/experiments/qwen3vl_2b_zero_shot_baseline.toml"

from shield.notebook_utils import (
    SYSTEM_PROMPT,
    ensure_dirs,
    evaluate_on_test_set,
    load_config,
    load_hf_dataset,
    load_json_records,
    print_dataset_summary,
    resolve_path,
    save_evaluation_results,
    set_seed,
    XRayDataCollator,
)

config = load_config(CONFIG_PATH, PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")
print(f"Config:       {CONFIG_PATH}")
print(f"Run:          {config['run']['name']}")

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Nessuna GPU disponibile. Verifica CUDA sul server.")

print(f"GPU disponibili: {torch.cuda.device_count()}")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {props.name} - {props.total_memory / 1e9:.1f} GB")

In [ ]:
processed_dir = resolve_path(config, "paths", "processed_dir")
checkpoints_dir = resolve_path(config, "paths", "checkpoints_dir")
evaluation_dir = resolve_path(config, "paths", "evaluation_dir")
model_dir = resolve_path(config, "paths", "model_dir")
ensure_dirs(checkpoints_dir, evaluation_dir)

train_dataset = load_hf_dataset(processed_dir / "train.json")
val_dataset = load_hf_dataset(processed_dir / "val.json")
test_records = load_json_records(processed_dir / "test.json")
print_dataset_summary(train_dataset, val_dataset, test_records)

In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

set_seed(config["inference"].get("seed", 42))

bnb_config = None
if config["inference"].get("load_in_4bit", True):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

print(f"Caricamento modello zero-shot: {model_dir}")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    str(model_dir),
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()

processor = AutoProcessor.from_pretrained(
    str(model_dir),
    min_pixels=256 * 28 * 28,
    max_pixels=config["inference"].get("max_image_pixels", 1003520),
)

print("Modello e processor pronti.")

In [ ]:
metrics_path = resolve_path(config, "evaluation", "metrics_file")
qualitative_path = resolve_path(config, "evaluation", "qualitative_file")

results = evaluate_on_test_set(
    model=model,
    processor=processor,
    test_records=test_records,
    label=config["run"]["name"],
    bertscore_model_type=config["evaluation"].get("bertscore_model_type", "xlm-roberta-large"),
    max_new_tokens=config["inference"].get("max_new_tokens", 512),
    qualitative_limit=config["evaluation"].get("qualitative_examples", 20),
)
save_evaluation_results(results, metrics_path, qualitative_path)

## Logging MLflow

Esegui questa cella solo se il server MLflow e' acceso.

In [ ]:
import subprocess

metrics_path = resolve_path(config, "evaluation", "metrics_file")
evaluation_dir = resolve_path(config, "paths", "evaluation_dir")
cmd = [
    "uv", "run", "python", "scripts/mlflow/log_run_to_mlflow.py",
    "--config", CONFIG_PATH,
    "--metrics", str(metrics_path.relative_to(PROJECT_ROOT)),
    "--artifact", str(evaluation_dir.relative_to(PROJECT_ROOT)),
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)